In [ ]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

In [ ]:
%cd capstone_project_GroupA
!git checkout main

In [ ]:
%cd src

In [25]:
# if you put this into another folder within src, u need below.
# import sys, os
# sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import glob
from NSWData.NSWDataLoader import *
from ModelFiles.ModelPlots import *
nsw_data_loader = NSWDataLoader()

Found NSW data path: C:\Users\kyim1\Desktop\capstone_project_GroupA\data\NSW


In [31]:
sarimax_folder = os.path.join(nsw_data_loader.output_dir, 'SARIMAX_results')
metrics_files = glob.glob(os.path.join(sarimax_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
sarimax_results_df = pd.concat(dfs, ignore_index=True)

gb_folder = os.path.join(nsw_data_loader.output_dir, 'gradient_boosting_results')
metrics_files = glob.glob(os.path.join(gb_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
gb_results_df = pd.concat(dfs, ignore_index=True)

lstm_folder = os.path.join(nsw_data_loader.output_dir, 'LSTM_results')
metrics_files = glob.glob(os.path.join(lstm_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
lstm_results_df = pd.concat(dfs, ignore_index=True)

patchtst_folder = os.path.join(nsw_data_loader.output_dir, 'PATCHTST_results')
metrics_files = glob.glob(os.path.join(patchtst_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
patchtst_results_df = pd.concat(dfs, ignore_index=True)

In [32]:
sarimax_results_df['model'] = 'SARIMAX'
sarimax_results_df['seed'] = '31415'
sarimax_results_df = sarimax_results_df[['model', 'seed', 'horizon','rmse']]
gb_results_df['model'] = 'Gradient Boosting'
gb_results_df = gb_results_df[['model', 'seed', 'horizon','rmse']]
lstm_results_df['model'] = 'LSTM'
lstm_results_df = lstm_results_df[['model', 'seed', 'horizon','rmse']]
patchtst_results_df['model'] = 'PATCHTST'
patchtst_results_df = patchtst_results_df[['model', 'seed', 'horizon','rmse']]
all_results_df = pd.concat([sarimax_results_df, gb_results_df, lstm_results_df, patchtst_results_df], ignore_index=True)

In [ ]:
all_results_df.groupby(['model', 'horizon'])['rmse'].agg(['mean', 'std', 'size']).reset_index()

,model,horizon,mean,std,size
0,Gradient Boosting,48,536.341414,0.009260,6
1,Gradient Boosting,336,655.670469,0.123945,6
2,Gradient Boosting,720,690.824622,0.169271,6
3,LSTM,48,564.984483,20.864307,6
4,LSTM,336,790.935147,17.104175,6
5,LSTM,720,858.991732,13.236071,6
6,PATCHTST,48,468.070516,3.867112,6
7,PATCHTST,336,637.350057,14.995018,6
8,PATCHTST,720,694.992606,19.408241,6
9,SARIMAX,48,544.645728,NaN,1


In [34]:
all_results_df = all_results_df.groupby(['model', 'horizon'])['rmse'].agg(['mean', 'std', 'size']).reset_index()
t_crit = all_results_df['size'].apply(lambda n: stats.t.ppf(0.975, df=n - 1))
se = all_results_df['std'] / all_results_df['size'] ** 0.5
all_results_df['CI_lower'] = all_results_df['mean'] - t_crit * se
all_results_df['CI_upper'] = all_results_df['mean'] + t_crit * se
# all_results_df['CI_lower'] = all_results_df['mean'] - 1.96 * all_results_df['std'] / (all_results_df['size'] ** 0.5)
# all_results_df['CI_upper'] = all_results_df['mean'] + 1.96 * all_results_df['std'] / (all_results_df['size'] ** 0.5)

In [35]:
all_results_df[['mean','std','CI_lower','CI_upper']] = all_results_df[['mean','std','CI_lower','CI_upper']].round(2)
print(all_results_df)

                model  horizon    mean    std  size  CI_lower  CI_upper
0   Gradient Boosting       48  536.34   0.01     6    536.33    536.35
1   Gradient Boosting      336  655.67   0.12     6    655.54    655.80
2   Gradient Boosting      720  690.82   0.17     6    690.65    691.00
3                LSTM       48  564.98  20.86     6    543.09    586.88
4                LSTM      336  790.94  17.10     6    772.99    808.88
5                LSTM      720  858.99  13.24     6    845.10    872.88
6            PATCHTST       48  468.07   3.87     6    464.01    472.13
7            PATCHTST      336  637.35  15.00     6    621.61    653.09
8            PATCHTST      720  694.99  19.41     6    674.62    715.36
9             SARIMAX       48  544.65    NaN     1       NaN       NaN
10            SARIMAX      336  650.79    NaN     1       NaN       NaN
11            SARIMAX      720  667.58    NaN     1       NaN       NaN
